# Dataset Preparation

## Introduction

This notebook focuses on the initial preparation of the CBIS-DDSM mammographic training and test datasets for subsequent deep learning experiments.

It extracts mammographic images from the original CBIS-DDSM directory structure and and associates each image with its corresponding diagnostic label, preparing the datasets for subsequent preprocessing and model development.

### Notebook Workflow

The notebook is structured into the following stages:

- extraction of all training and test image files from their original directory structures;
- creation of dedicated training and test directories containing consistently renamed image files;
- labeling of each image file according to the corresponding clinical diagnosis;
- final datasets integrity checks for downstream preprocessing.

### Python Modules

The following Python libraries are used throughout the notebook:

- os for filesystem navigation and file management;
- pandas for tabular data and metadata handling;
- multiprocessing for parallel image extraction and processing.

## Dataset Setup

This section imports the required Python libraries and defines the main paths used for dataset access and processing.

Note: The actual CSV files and mammographic images are not included in this repository. The directory names used in this repository differ slightly from the original CBIS-DDSM structure to clearly separate the training and test datasets. The required dataset files can be obtained from the original source [The Cancer Imaging Archive - CBIS-DDSM](https://www.cancerimagingarchive.net/collection/cbis-ddsm/).

In [ ]:
# Import required libraries
from pathlib import Path
import os
import pandas as pd
import multiprocessing as mp

# Dataset paths
BASE_DIR = Path("..")

DATA_PATH = BASE_DIR / "data"
RAW_DATA_PATH = DATA_PATH / "raw"
PROCESSED_DATA_PATH = DATA_PATH / "processed"

## Training Image Extraction

The CBIS-DDSM dataset stores mammographic images within a nested directory structure.

This section extracts training images from the CBIS-DDSM folder structure and renames them using their original folder names.

In [ ]:
# Define train dataset source and destination directories
train_folder = RAW_DATA_PATH / "CBIS-DDSM-TRAIN"
train_dest = PROCESSED_DATA_PATH / "Train"

# Change working directory
os.chdir(train_folder)

# List all subfolders in the train directory
study_folders = os.listdir()

# Function to extract train images
def create_train_set(m):
    global study_folders, train_dest

    # Navigate through nested folder structure
    os.chdir(study_folders[m])
    date_folder = os.listdir()

    os.chdir(date_folder[0])
    series_folder = os.listdir()

    os.chdir(series_folder[0])
    image_file = os.listdir()

    # Existence check for moving and renaming file
    if image_file:
        os.rename(image_file[0], train_dest / study_folders[m])

# Parallel extraction of train images
with mp.Pool() as train_pool:
    train_pool.map(create_train_set, range(len(study_folders)))

The goal of this section is to efficiently create a folder containing all training image files renamed using their original folder names.

## Test Image Extraction

Similarly to the preceding section, this section extracts test images from the CBIS-DDSM folder structure and renames them using their original folder names.

In [ ]:
# Define test dataset source and destination directories
test_folder = RAW_DATA_PATH / "CBIS-DDSM-TEST"
test_dest = PROCESSED_DATA_PATH / "Test"

# Change working directory
os.chdir(test_folder)

# List all subfolders in the test directory
study_folders = os.listdir()

# Function to extract test images
def create_test_set(m):
    global study_folders, test_dest

    # Navigate through nested folder structure
    os.chdir(study_folders[m])
    date_folder = os.listdir()

    os.chdir(date_folder[0])
    series_folder = os.listdir()

    os.chdir(series_folder[0])
    image_file = os.listdir()

    # Existence check for moving and renaming file
    if image_file:
        os.rename(image_file[0], test_dest / study_folders[m])

# Parallel extraction of test images
with mp.Pool() as test_pool:
    test_pool.map(create_test_set, range(len(study_folders)))

The goal of this section is to efficiently create a folder containing all test image files renamed using their original folder names.

## Training Image Labeling

This section assigns a binary diagnostic label to each training image according to the corresponding clinical diagnosis reported in the CBIS-DDSM training mass case CSV file.

Each mammogram is labeled as benign (0) or malignant (1) according to the associated lesion diagnosis and the assigned label is appended directly to its filename.

The tabular file describes each individual breast lesion, but a single mammographic image may show multiple masses with different diagnoses. In such cases, a conservative labeling strategy is adopted: the image is labeled as malignant (1) if at least one associated lesion is malignant.

In [ ]:
# Read train data file
train_data = pd.read_csv(filepath_or_buffer = RAW_DATA_PATH /
                         "mass_case_description_train_set.csv",
                         engine = 'python')

# Change working directory to the train dataset
os.chdir(train_dest)

# List all train image files
train_files = os.listdir()

# Function to assign binary labels to train images
def label_train_set(i):
    global train_files, train_data

    filename = train_files[i]

    # Skip already labeled file
    if not filename.endswith(('_0.png', '_1.png')):
        listed_name = list(filename)
        name_root = filename[:-4]
        counter = 0

        # Search corresponding tabular row
        while counter < len(train_data) and name_root not in train_data.loc[counter, 'image file path']:
            counter += 1

        if counter == len(train_data):
            return

        while name_root in train_data.loc[counter, 'image file path']:

            # Pathology check for labeling file
            if train_data.loc[counter, 'pathology'] == 'MALIGNANT':
                listed_name.insert(-4, '_1')
                break

            elif counter + 1 >= len(train_data) or name_root not in train_data.loc[counter + 1, 'image file path']:
                listed_name.insert(-4, '_0')
                break

            else:
                counter += 1

        os.rename(train_files[i], "".join(listed_name))

# Parallel labeling of train images
with mp.Pool() as label_train_pool:
    label_train_pool.map(label_train_set, range(len(train_files)))

At the end of this section, the training set is fully labeled with binary diagnostic classes and ready for the subsequent preprocessing and training stages.

## Test Image Labeling

This section assigns a binary diagnostic label to each test image according to the corresponding clinical diagnosis reported in the CBIS-DDSM test mass case CSV file, following the same approach described in the previous section.

The assigned label is then appended directly to the image filename using the same naming convention adopted for the training set.

In cases where a single mammogram is associated with multiple lesions having different diagnoses, the image is labeled as malignant (1) if at least one lesion is malignant.

In [ ]:
# Read test data file
test_data = pd.read_csv(filepath_or_buffer = RAW_DATA_PATH /
                         "mass_case_description_test_set.csv",
                         engine = 'python')

# Change working directory to the test dataset
os.chdir(test_dest)

# List all test image files
test_files = os.listdir()

# Function to assign binary labels to test images
def label_test_set(i):
    global test_files, test_data

    filename = test_files[i]

    # Skip already labeled file
    if not filename.endswith(('_0.png', '_1.png')):
        listed_name = list(filename)
        name_root = filename[:-4]
        counter = 0

        # Search corresponding tabular row
        while counter < len(test_data) and name_root not in test_data.loc[counter, 'image file path']:
            counter += 1

        if counter == len(test_data):
            return

        while name_root in test_data.loc[counter, 'image file path']:

            # Pathology check for labeling file
            if test_data.loc[counter, 'pathology'] == 'MALIGNANT':
                listed_name.insert(-4, '_1')
                break

            elif counter + 1 >= len(test_data) or name_root not in test_data.loc[counter + 1, 'image file path']:
                listed_name.insert(-4, '_0')
                break

            else:
                counter += 1

        os.rename(test_files[i], "".join(listed_name))

# Parallel labeling of test images
with mp.Pool() as label_test_pool:
    label_test_pool.map(label_test_set, range(len(test_files)))

At the end of this section, the test set is fully labeled with binary diagnostic classes and ready for subsequent model testing and evaluation.